# Fine-Grained Access Control with Bedrock AgentCore Gatway Interceptors using Data Store

## Overview

This notebook shows how to enforce **Fine-Grained Access Control (FGAC)** on an **AgentCore Gateway** using **Gateway interceptors** and **Cognito scopes**. The goal is to give you a reusable pattern to protect your agent endpoints, no matter how many tools your MCP target provides.

### Why This Matters

As your agent expands, you may need to:

- Restrict which **tools** certain users can call  
- Control access to **sensitive actions** (cancelOrder, updateOrder, deleteOrder, etc.)  
- Filter or redact **semantic search results** based on user permissions  
- Show users **only the tools they are allowed to see**  
- Enforce custom authorization logic that goes beyond what JWT tokens provide  
- Apply **centralized governance** without modifying individual tools or runtimes  

Gateway Interceptors provide a scalable, plug-and-play way to implement these controls **without modifying the agent, the runtime, or the MCP server**.  
You enforce policy **at the Gateway level**, where every request naturally flows through.

---

## What This Tutorial Covers

You will implement FGAC for various tools with Gateway operations:

 📋 **List Tools with FGAC (RESPONSE interceptor)**  
   Dynamically filter the tool catalog so users only see tools they’re authorized for.  
   ![list tool](../images/FGAC_data_store.png)

---

## Why Use Gateway Interceptors?

Gateway Interceptors allow you to:

- **Implement Fine-Grained Access Control**  
  Enforce per-user, per-tool, per-action authorization rules.

- **Inject Custom Authorization Logic**  
  Go beyond static JWT validation with dynamic rules or external policies.

- **Audit & Governance**  
  Log attempted tool usage and provide compliance visibility.

- **Request/Response Transformation**  
  Redact data, modify requests, or filter responses before users see them.

Because interceptors are attached at the **Gateway layer**, they enforce central policy for **any** underlying MCP server or Runtime.

---

## Tutorial Details

| Information              | Details                                                                                         |
|--------------------------|-------------------------------------------------------------------------------------------------|
| **Tutorial type**        | Interactive                                                                                     |
| **AgentCore components** | AgentCore Gateway, Gateway Interceptors                                              |
| **Gateway Target type**  | MCP Server (FastMCP running on AgentCore Runtime)                                              |
| **Interceptor types**    | AWS Lambda (REQUEST + RESPONSE)                                                                |
| **Inbound Auth IdP**     | Amazon Cognito (CUSTOM\_JWT authorizer)                                                        |
| **DataStore**            | Amazon DynamoDB (for tool mappings)                                                      |
| **Access Control**       | FGAC using Cognito scopes + Gateway interceptors                                                |
| **Tutorial components**  | Gateway, Runtime MCP Server, Cognito, Gateway Interceptors, MCP tools                           |
| **Tutorial vertical**    | Cross-vertical                                                                                  |
| **Example complexity**   | Easy–Intermediate                                                                              |
| **SDK used**             | boto3                                                                                           |

---

## Prerequisites

To execute this tutorial you will need:

- Jupyter notebook (Python kernel)
- AWS credentials with permissions for:
  - Lambda
  - IAM
  - Cognito
  - DynamoDB
  - AgentCore services (control plane + runtime)
- Python 3.13 or higher
- Basic understanding of AWS Lambda, IAM roles, Cognito, and AgentCore Gateway

> ⚠️ **Note:** The Cleanup section at the end deletes the AWS resources created by this tutorial (Gateway, Lambdas, IAM roles, etc.). Only run it when you’re ready to tear everything down.


In [ ]:
# Import required libraries
import boto3
import json
import time
import zipfile
import io
import requests
from pathlib import Path
from datetime import datetime
from botocore.exceptions import ClientError
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

print("✓ Libraries imported")

# Generate unique identifier for this deployment
DEPLOYMENT_ID = datetime.now().strftime('%Y%m%d-%H%M%S')
print(f"\nDeployment ID: {DEPLOYMENT_ID}")

# Configuration
DYNAMODB_TABLE_NAME = "ClientToolPermissions"
DYNAMODB_REGION = "us-east-1"
LAMBDA_REGION = "us-east-1"  # Gamma is in us-east-1


# Resource names
LAMBDA_FUNCTION_NAME = f"interceptor-lambda-{DEPLOYMENT_ID}"
LAMBDA_ROLE_NAME = f"interceptor-lambda-role-{DEPLOYMENT_ID}"
GATEWAY_NAME = f"interceptor-gateway-{DEPLOYMENT_ID}"

print("Configuration:")
print(f"  DynamoDB Table: {DYNAMODB_TABLE_NAME}")
print(f"  Lambda Function: {LAMBDA_FUNCTION_NAME}")
print(f"  Lambda Role: {LAMBDA_ROLE_NAME}")
print(f"  Gateway Name: {GATEWAY_NAME}")
print(f"  Region: {LAMBDA_REGION}")

---

## Part 1: Setup & Deployment

### Step 1.1: Create Cognito User Pool & App Clients
Create multiple app clients with different permission levels.


In [ ]:
# Create Cognito User Pool and Client for Gateway authentication
print("Creating Cognito User Pool and Client...")

cognito_client = boto3.client('cognito-idp', region_name='us-east-1')

# Create User Pool
USER_POOL_NAME = f"gateway-pool-{DEPLOYMENT_ID}"

try:
    pool_response = cognito_client.create_user_pool(
        PoolName=USER_POOL_NAME,
        Policies={'PasswordPolicy': {
            'MinimumLength': 8,
            'RequireUppercase': False,
            'RequireLowercase': False,
            'RequireNumbers': False,
            'RequireSymbols': False
        }}
    )
    
    USER_POOL_ID = pool_response['UserPool']['Id']
    print(f"✓ User Pool created: {USER_POOL_NAME}")
    print(f"  Pool ID: {USER_POOL_ID}")
    
except ClientError as e:
    print(f"⚠ Error creating user pool: {e}")
    raise

# Create User Pool Domain (required for OAuth)
POOL_DOMAIN = f"interceptor-{DEPLOYMENT_ID.replace('_', '-').lower()}"

try:
    cognito_client.create_user_pool_domain(Domain=POOL_DOMAIN, UserPoolId=USER_POOL_ID)
    print(f"✓ User Pool Domain created: {POOL_DOMAIN}")
except ClientError as e:
    if 'Domain already exists' in str(e):
        print(f"⚠ Domain already exists: {POOL_DOMAIN}")
    else:
        print(f"⚠ Error creating domain: {e}")

# Create Resource Server with custom scope
try:
    cognito_client.create_resource_server(
        UserPoolId=USER_POOL_ID,
        Identifier='gateway',
        Name='Gateway Resource Server',
        Scopes=[{'ScopeName': 'tools', 'ScopeDescription': 'Access to gateway tools'}]
    )
    print(f"✓ Resource Server created with scope: gateway/tools")
except ClientError as e:
    print(f"⚠ Resource server error: {e}")

# Wait for resource server
print("  Waiting for resource server to propagate...")
time.sleep(3)

# Create User Pool Client with client credentials flow
CLIENT_NAME = f"gateway-client-{DEPLOYMENT_ID}"

try:

        # Create multiple app clients for different permission levels
    client_configs = [
        'full-access',
        'readonly',
        'calculator',
        'data'
    ]

    clients = {}

    for client_type in client_configs:
        response = cognito_client.create_user_pool_client(
            UserPoolId=USER_POOL_ID,
            ClientName=f"{client_type}-client-{DEPLOYMENT_ID}",
            GenerateSecret=True,
            ExplicitAuthFlows=[],
            AllowedOAuthFlows=['client_credentials'],
            AllowedOAuthScopes=['gateway/tools'],
            AllowedOAuthFlowsUserPoolClient=True,
            SupportedIdentityProviders=[]
        )
        
        clients[client_type] = {
            'client_id': response['UserPoolClient']['ClientId'],
        }


    # Easy access to client IDs
    CLIENT_ID_FULL = clients['full-access']['client_id']
    CLIENT_ID_READONLY = clients['readonly']['client_id']
    CLIENT_ID_CALCULATOR = clients['calculator']['client_id']
    CLIENT_ID_DATA = clients['data']['client_id']
    
    # Construct OAuth URLs
    COGNITO_DOMAIN = f"https://{POOL_DOMAIN}.auth.us-east-1.amazoncognito.com"
    DISCOVERY_URL = f"https://cognito-idp.us-east-1.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"
    TOKEN_URL = f"{COGNITO_DOMAIN}/oauth2/token"
    
    print(f"\n✓ OAuth Configuration:")
    print(f"  Discovery URL: {DISCOVERY_URL}")
    print(f"  Token URL: {TOKEN_URL}")
    print(f"  Scope: gateway/tools")
    
except ClientError as e:
    print(f"✗ Error creating client: {e}")
    raise

### Step 1.2: Create DynamoDB Permissions Table
Create table with ClientID (partition key)

In [ ]:
# Create DynamoDB table
print("Creating DynamoDB table...")

dynamodb_client = boto3.client('dynamodb', region_name=DYNAMODB_REGION)

try:
    response = dynamodb_client.create_table(
        TableName=DYNAMODB_TABLE_NAME,
        KeySchema=[
            {'AttributeName': 'ClientID', 'KeyType': 'HASH'},
            {'AttributeName': 'ToolName', 'KeyType': 'RANGE'}
        ],
        AttributeDefinitions=[
            {'AttributeName': 'ClientID', 'AttributeType': 'S'},
            {'AttributeName': 'ToolName', 'AttributeType': 'S'}
        ],
        BillingMode='PAY_PER_REQUEST'
    )
    
    print(f"✓ Table created: {DYNAMODB_TABLE_NAME}")
    print(f"  Status: {response['TableDescription']['TableStatus']}")
    
    # Wait for table to be active
    print("  Waiting for table to be active...")
    waiter = dynamodb_client.get_waiter('table_exists')
    waiter.wait(TableName=DYNAMODB_TABLE_NAME)
    print("  ✓ Table is active")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceInUseException':
        print(f"⚠ Table already exists: {DYNAMODB_TABLE_NAME}")
    else:
        print(f"✗ Error: {e}")
        raise

### Step 1.3: Load Client Permissions into DynamoDB
Map each Cognito client_id to their allowed tools.

In [ ]:
# Load sample permissions into DynamoDB
print("Loading sample permissions...")

dynamodb = boto3.resource('dynamodb', region_name=DYNAMODB_REGION)
table = dynamodb.Table(DYNAMODB_TABLE_NAME)

# Sample permissions using example client IDs
# NOTE: Replace these with your actual Cognito client_id values from JWT tokens
SAMPLE_PERMISSIONS = [
    # Example client 1: Full access (replace with your actual client_id)
    {'ClientID': CLIENT_ID_FULL, 'ToolName': 'weather_tool', 'Allowed': True},
    {'ClientID': CLIENT_ID_FULL, 'ToolName': 'database_query_tool', 'Allowed': True},
    {'ClientID': CLIENT_ID_FULL, 'ToolName': 'calculation_tool', 'Allowed': True},
    {'ClientID': CLIENT_ID_FULL, 'ToolName': 'search_tool', 'Allowed': True},
    {'ClientID': CLIENT_ID_FULL, 'ToolName': 'file_handler_tool', 'Allowed': True},
    
    # Example client 2: Read-only access (weather and search only)
    {'ClientID': CLIENT_ID_READONLY, 'ToolName': 'weather_tool', 'Allowed': True},
    {'ClientID': CLIENT_ID_READONLY, 'ToolName': 'search_tool', 'Allowed': True},
    
    # Example client 3: Calculator only
    {'ClientID': CLIENT_ID_CALCULATOR, 'ToolName': 'calculation_tool', 'Allowed': True},
    
    # Example client 4: Data processing (database, file, calculation)
    {'ClientID': CLIENT_ID_DATA, 'ToolName': 'database_query_tool', 'Allowed': True},
    {'ClientID': CLIENT_ID_DATA, 'ToolName': 'file_handler_tool', 'Allowed': True},
    {'ClientID': CLIENT_ID_DATA, 'ToolName': 'calculation_tool', 'Allowed': True},
]

# Batch write
with table.batch_writer() as batch:
    for perm in SAMPLE_PERMISSIONS:
        perm['CreatedAt'] = datetime.utcnow().isoformat()
        perm['UpdatedAt'] = datetime.utcnow().isoformat()
        batch.put_item(Item=perm)

print(f"✓ Loaded {len(SAMPLE_PERMISSIONS)} permissions")

# Count clients
clients = set(p['ClientID'] for p in SAMPLE_PERMISSIONS)
print(f"  Configured clients: {len(clients)}")
for client in sorted(clients):
    count = len([p for p in SAMPLE_PERMISSIONS if p['ClientID'] == client])
    print(f"    • {client}: {count} tools")

### Step 1.4: Create IAM Role for Lambda Interceptor
Grant Lambda permissions to read DynamoDB and write CloudWatch logs.

In [ ]:
# Create IAM role for Lambda
print("Creating IAM role...")

iam_client = boto3.client('iam')

# Trust policy for Lambda
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}

try:
    role_response = iam_client.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Role for AgentCore Lambda Interceptor'
    )
    
    LAMBDA_ROLE_ARN = role_response['Role']['Arn']
    print(f"✓ IAM Role created: {LAMBDA_ROLE_NAME}")
    print(f"  ARN: {LAMBDA_ROLE_ARN}")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        print(f"⚠ Role already exists: {LAMBDA_ROLE_NAME}")
        role_response = iam_client.get_role(RoleName=LAMBDA_ROLE_NAME)
        LAMBDA_ROLE_ARN = role_response['Role']['Arn']
        print(f"  ARN: {LAMBDA_ROLE_ARN}")
    else:
        raise

# Attach basic Lambda execution policy
print("  Attaching Lambda execution policy...")
iam_client.attach_role_policy(
    RoleName=LAMBDA_ROLE_NAME,
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
)

# Create and attach DynamoDB policy
print("  Creating DynamoDB access policy...")

table_arn = f"arn:aws:dynamodb:{DYNAMODB_REGION}:{boto3.client('sts').get_caller_identity()['Account']}:table/{DYNAMODB_TABLE_NAME}"

dynamodb_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "dynamodb:Query",
                "dynamodb:GetItem"
            ],
            "Resource": table_arn
        }
    ]
}

try:
    iam_client.put_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyName='DynamoDBAccess',
        PolicyDocument=json.dumps(dynamodb_policy)
    )
    print("  ✓ DynamoDB policy attached")
except Exception as e:
    print(f"  ⚠ Policy error: {e}")

# Wait for role to propagate
print("  Waiting for IAM role to propagate...")
time.sleep(10)
print("  ✓ IAM role ready")

### Step 1.5: Deploy Lambda Interceptor Function
Lambda extracts client_id from JWT and filters tools based on DynamoDB permissions.

In [ ]:
# Verify Lambda code uses correct format
print("Verifying Lambda interceptor code...")

lambda_code_path = Path('src/lambda/lambda_function.py')

if not lambda_code_path.exists():
    print(f"✗ Lambda code not found: {lambda_code_path}")
    raise FileNotFoundError(f"Missing {lambda_code_path}")

with open(lambda_code_path, 'r') as f:
    lambda_code = f.read()

# Verify it uses the correct response format
if '"interceptorOutputVersion": "1.0"' in lambda_code:
    print("✓ Lambda code uses CORRECT response format (interceptorOutputVersion)")
elif '"customInterceptorOutputVersion": "1.0"' in lambda_code:
    print("✗ ERROR: Lambda code uses OLD format (customInterceptorOutputVersion)")
    print("  Please update src/lambda/lambda_function.py")
    print("  Change ALL occurrences:")
    print("    customInterceptorOutputVersion → interceptorOutputVersion")
    raise Exception("Lambda code needs update - uses deprecated format")
else:
    print("⚠ Could not verify response format in Lambda code")

print(f"✓ Lambda code verified: {len(lambda_code)} bytes")

# Create deployment package
print("Creating deployment package...")
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zip_file:
    zip_file.writestr('lambda_function.py', lambda_code)

zip_buffer.seek(0)
deployment_package = zip_buffer.read()
print(f"✓ Package size: {len(deployment_package)} bytes")

In [ ]:
# Create Lambda function
print("Creating Lambda function...")

lambda_client = boto3.client('lambda', region_name=LAMBDA_REGION)

try:
    response = lambda_client.create_function(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Runtime='python3.9',
        Role=LAMBDA_ROLE_ARN,
        Handler='lambda_function.lambda_handler',
        Code={'ZipFile': deployment_package},
        Description='AgentCore Lambda Interceptor - Filters tools based on DynamoDB permissions',
        Timeout=30,
        MemorySize=256,
        Environment={
            'Variables': {
                'PERMISSIONS_TABLE_NAME': DYNAMODB_TABLE_NAME,
                'DYNAMODB_REGION': DYNAMODB_REGION
            }
        }
    )
    
    LAMBDA_ARN = response['FunctionArn']
    print(f"✓ Lambda created: {LAMBDA_FUNCTION_NAME}")
    print(f"  ARN: {LAMBDA_ARN}")
    print(f"  Runtime: {response['Runtime']}")
    print(f"  Memory: {response['MemorySize']} MB")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceConflictException':
        print(f"⚠ Lambda already exists: {LAMBDA_FUNCTION_NAME}")
        response = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)
        LAMBDA_ARN = response['Configuration']['FunctionArn']
        print(f"  ARN: {LAMBDA_ARN}")
    else:
        raise

In [ ]:
# ⭐ CRITICAL: Grant Gateway permission to invoke the interceptor Lambda
print("\n🔐 Granting Gateway permission to invoke Lambda...")

# Get AWS account ID
sts_client = boto3.client('sts')
ACCOUNT_ID = sts_client.get_caller_identity()['Account']

try:
    lambda_client.add_permission(
        FunctionName=LAMBDA_FUNCTION_NAME,
        StatementId='AllowGatewayInvoke',
        Action='lambda:InvokeFunction',
        Principal='bedrock-agentcore.amazonaws.com',
        SourceArn=f'arn:aws:bedrock-agentcore:{LAMBDA_REGION}:{ACCOUNT_ID}:gateway/*'
    )
    print(f"✓ Gateway invoke permission added to Lambda")
    print(f"  Principal: bedrock-agentcore.amazonaws.com")
    print(f"  Source: arn:aws:bedrock-agentcore:{LAMBDA_REGION}:{ACCOUNT_ID}:gateway/*")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceConflictException':
        print(f"⚠ Permission already exists (this is fine)")
    else:
        print(f"⚠ Error adding permission: {e}")
        raise


### Step 1.6: Create Gateway with Response Interceptor
Configure Gateway to use Lambda interceptor for tools/list responses.

In [ ]:
# Initialize Boto3 client for bedrock-agentcore-control
print("Initializing Boto3 Gateway client...")

gateway_client = boto3.client('bedrock-agentcore-control', region_name=LAMBDA_REGION)

print(f"✓ Gateway client initialized for region: {LAMBDA_REGION}")

In [ ]:
# Create Gateway with Lambda interceptor using signed HTTP requests
print("Creating Gateway with Lambda RESPONSE interceptor...")

# First, create an IAM role for the Gateway
iam_client = boto3.client('iam')

    
gateway_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock-agentcore.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

GATEWAY_ROLE_NAME = f"gateway-role-{DEPLOYMENT_ID}"

try:
    gateway_role_response = iam_client.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(gateway_trust_policy),
        Description='IAM role for Bedrock AgentCore Gateway'
    )
    
    GATEWAY_ROLE_ARN = gateway_role_response['Role']['Arn']
    print(f"✓ Gateway IAM role created: {GATEWAY_ROLE_NAME}")
    print(f"  ARN: {GATEWAY_ROLE_ARN}")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        print(f"⚠ Role already exists: {GATEWAY_ROLE_NAME}")
        gateway_role_response = iam_client.get_role(RoleName=GATEWAY_ROLE_NAME)
        GATEWAY_ROLE_ARN = gateway_role_response['Role']['Arn']
        print(f"  ARN: {GATEWAY_ROLE_ARN}")
    else:
        raise

# Attach necessary policies to the Gateway role
print("  Attaching policies to Gateway role...")
try:
    # Attach Lambda invoke policy
    lambda_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": "lambda:InvokeFunction",
                "Resource": "*"
            }
        ]
    }
    
    iam_client.put_role_policy(
        RoleName=GATEWAY_ROLE_NAME,
        PolicyName='LambdaInvokePolicy',
        PolicyDocument=json.dumps(lambda_policy)
    )
    print("  ✓ Lambda invoke policy attached")
except Exception as e:
    print(f"  ⚠ Policy attach error: {e}")

# Wait for Gateway role to propagate
print("  Waiting for Gateway role to propagate...")
time.sleep(10)

# Create Gateway using signed HTTP request with interceptor configuration
print(f"\n  Creating Gateway with RESPONSE interceptor:")
print(f"    Name: {GATEWAY_NAME}")
print(f"    Protocol: MCP")
print(f"    Auth: CUSTOM_JWT (Cognito)")
print(f"    Interceptor: {LAMBDA_ARN}")

try:
    gateway_response = gateway_client.create_gateway(
    name=GATEWAY_NAME,
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {
            "supportedVersions": ["2025-03-26"]
        }
    },

    interceptorConfigurations=[
        {
            "interceptor": {
                "lambda": {
                    "arn": LAMBDA_ARN
                }
            },
            "interceptionPoints": ["RESPONSE"],  # Intercept responses to filter tools
            "inputConfiguration": {
                "passRequestHeaders": True  
            }
        }
    ],

    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": DISCOVERY_URL,
            "allowedClients": [CLIENT_ID_FULL, CLIENT_ID_DATA, CLIENT_ID_CALCULATOR,CLIENT_ID_READONLY]
        }
    },

    roleArn=GATEWAY_ROLE_ARN
)
    status_code = gateway_response.get("ResponseMetadata", {}).get("HTTPStatusCode")
    if status_code not in [200, 202]:
        print(f"\n✗ Failed to create Gateway: {status_code}")
        print(f"  Response: {gateway_response.text}")
        raise Exception(f"Gateway creation failed: {gateway_response.text}")
    
    # gateway_data = gateway_response.json()
    GATEWAY_ID = gateway_response.get('gatewayId')
    
    print(f"\n✓ Gateway created successfully with RESPONSE interceptor")
    print(f"  ID: {GATEWAY_ID}")
    print(f"  Status: {gateway_response.get('status', 'CREATING')}")
    print(f"  Interceptor Lambda: {LAMBDA_ARN}")
    print(f"  Interception Point: RESPONSE (filters tools after aggregation)")
    
    # Verify interceptor configuration in response
    if 'interceptorConfigurations' in gateway_response and gateway_response['interceptorConfigurations']:
        print(f"  ✓ Interceptor configuration confirmed in response!")
        print(f"    Interceptors: {len(gateway_response['interceptorConfigurations'])}")
    
except Exception as e:
    print(f"\n✗ Failed to create Gateway: {e}")
    raise


In [ ]:
# Wait for Gateway to be ready using signed requests
print("\nWaiting for Gateway to be ready...")

max_attempts = 30
for attempt in range(max_attempts):
    try:
        response = gateway_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
        status_code = response.get("ResponseMetadata", {}).get("HTTPStatusCode")

        if status_code == 200:
            status = response.get('status', 'UNKNOWN')
            
            print(f"  [{attempt + 1}/{max_attempts}] Status: {status}")
            
            if status == 'READY':
                GATEWAY_URL = response.get('gatewayUrl')
                print(f"\n✓ Gateway is ready!")
                print(f"  URL: {GATEWAY_URL}")
                
                # Show interceptor configuration
                if 'interceptorConfigurations' in response:
                    interceptor_configs = response['interceptorConfigurations']
                    print(f"\n  Interceptor Configuration:")
                    for idx, config in enumerate(interceptor_configs):
                        print(f"    [{idx}] Interception Points: {config.get('interceptionPoints', [])}")
                        print(f"    [{idx}] Lambda ARN: {config.get('interceptor', {}).get('lambda', {}).get('arn', 'N/A')}")
                        print(f"    [{idx}] Pass Headers: {config.get('inputConfiguration', {}).get('passRequestHeaders', False)}")
                break
            elif status == 'FAILED':
                print(f"\n✗ Gateway creation failed")
                print(f"  Details: {response}")
                raise Exception("Gateway failed")
        else:
            print(f"  [{attempt + 1}/{max_attempts}] HTTP Error: {response.status_code}")
    except Exception as e:
        print(f"  [{attempt + 1}/{max_attempts}] Error: {e}")
    
    time.sleep(10)
else:
    print(f"\n⚠ Timeout waiting for Gateway")
    raise Exception("Gateway timeout")


In [ ]:
# ⭐ CRITICAL: Verify interceptor configuration is actually stored
print("\n🔍 Verifying interceptor configuration on Gateway...")
print("-" * 60)

if 'interceptorConfigurations' not in response:
    print("\n❌ ERROR: Gateway does NOT have interceptor configured!")
    print("The interceptorConfigurations parameter was not accepted.")
    print("\nGateway Info:")
    print(json.dumps(response, indent=2, default=str))
    raise Exception("Interceptor not configured on Gateway - this explains why filtering isn't working!")

# If we get here, interceptor IS configured
interceptor_configs = response['interceptorConfigurations']
print(f"✓ Interceptor configuration found!")
print(f"  Number of interceptors: {len(interceptor_configs)}")

for idx, config in enumerate(interceptor_configs):
    print(f"\n  Interceptor [{idx}]:")
    print(f"    Interception Points: {config.get('interceptionPoints', [])}")
    print(f"    Lambda ARN: {config.get('interceptor', {}).get('lambda', {}).get('arn', 'N/A')}")
    print(f"    Pass Headers: {config.get('inputConfiguration', {}).get('passRequestHeaders', False)}")
    
    # Verify it matches what we configured
    configured_arn = config.get('interceptor', {}).get('lambda', {}).get('arn', '')
    if configured_arn == LAMBDA_ARN:
        print(f"    ✓ Lambda ARN matches our interceptor")
    else:
        print(f"    ⚠ Lambda ARN mismatch!")
        print(f"      Expected: {LAMBDA_ARN}")
        print(f"      Got: {configured_arn}")
    
    # Verify passRequestHeaders is enabled
    pass_headers = config.get('inputConfiguration', {}).get('passRequestHeaders', False)
    if pass_headers:
        print(f"    ✓ Request headers will be passed to interceptor")
    else:
        print(f"    ⚠ WARNING: passRequestHeaders is FALSE - Agent-ID won't be passed!")

print("\n" + "-" * 60)
print("✅ Interceptor configuration verification complete")


### Step 1.7: Register Sample Tools with Gateway
Deploy tool Lambdas and register them as Gateway targets.

In [ ]:
# Deploy real tool Lambdas and register as Gateway targets using SIGNED HTTP requests
print("="*80)
print("Deploying Real Tool Lambdas and Registering with Gateway")
print("="*80)

import sys
import importlib

# Step 1: Deploy tool Lambda functions
print("\n📦 Step 1: Deploying tool Lambda functions...")
print("-" * 60)

# Import tool modules with RELOAD to get latest changes
sys.path.insert(0, str(Path.cwd()))

# Force reload modules to get updated TOOL_DEFINITION (without enum)
from src.tools import weather_tool, database_query_tool, calculation_tool, search_tool, file_handler_tool

# Reload all modules to ensure we get the latest TOOL_DEFINITION
weather_tool = importlib.reload(weather_tool)
database_query_tool = importlib.reload(database_query_tool)
calculation_tool = importlib.reload(calculation_tool)
search_tool = importlib.reload(search_tool)
file_handler_tool = importlib.reload(file_handler_tool)

print("✓ Tool modules reloaded with updated schemas")

# Create IAM role for tool Lambdas
TOOL_ROLE_NAME = f"tool-lambda-role-{DEPLOYMENT_ID}"

try:
    tool_role_response = iam_client.create_role(
        RoleName=TOOL_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
        }),
        Description='Role for tool Lambda functions'
    )
    TOOL_ROLE_ARN = tool_role_response['Role']['Arn']
    print(f"✓ Tool IAM role created: {TOOL_ROLE_NAME}")
    
    # Attach basic execution policy
    iam_client.attach_role_policy(
        RoleName=TOOL_ROLE_NAME,
        PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
    )
    time.sleep(15)  # Wait for role to propagate
    
except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        tool_role_response = iam_client.get_role(RoleName=TOOL_ROLE_NAME)
        TOOL_ROLE_ARN = tool_role_response['Role']['Arn']
        print(f"⚠ Tool role already exists: {TOOL_ROLE_NAME}")
    else:
        raise

# Additional wait to ensure IAM is fully propagated
print("⏳ Waiting 15 seconds for IAM role propagation...")
time.sleep(15)
print("✓ IAM role should be ready")

# Deploy each tool Lambda
tools_to_deploy = [
    ('weather_tool', weather_tool),
    ('database_query_tool', database_query_tool),
    ('calculation_tool', calculation_tool),
    ('search_tool', search_tool),
    ('file_handler_tool', file_handler_tool),
]

deployed_tools = []

for tool_name, tool_module in tools_to_deploy:
    print(f"\n  Deploying {tool_name}...")
    
    # Create ZIP
    tool_code_path = Path(tool_module.__file__)
    with open(tool_code_path, 'r') as f:
        tool_code = f.read()
    
    zip_buf = io.BytesIO()
    with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr('lambda_function.py', tool_code)
    zip_buf.seek(0)
    
    function_name = f"{tool_name.replace('_', '-')}-{DEPLOYMENT_ID}"
    
    try:
        response = lambda_client.create_function(
            FunctionName=function_name,
            Runtime='python3.9',
            Role=TOOL_ROLE_ARN,
            Handler='lambda_function.lambda_handler',
            Code={'ZipFile': zip_buf.read()},
            Timeout=30,
            MemorySize=256,
            Environment={'Variables': {'TOOL_NAME': tool_name}}
        )
        lambda_arn = response['FunctionArn']
        print(f"    ✓ Created: {function_name}")
        
    except ClientError as e:
        if e.response['Error']['Code'] == 'ResourceConflictException':
            response = lambda_client.get_function(FunctionName=function_name)
            lambda_arn = response['Configuration']['FunctionArn']
            print(f"    ⚠ Already exists: {function_name}")
        else:
            raise
    
    # Get tool definition (now reloaded without enum)
    tool_definition = getattr(tool_module, 'TOOL_DEFINITION', {
        "name": tool_name,
        "description": f"{tool_name} function"
    })
    
    # Verify no enum in tool_definition
    tool_def_str = json.dumps(tool_definition)
    if '"enum"' in tool_def_str:
        print(f"    ⚠ WARNING: Tool definition still contains 'enum' - module may not have reloaded!")
        print(f"    Tool definition: {tool_def_str[:200]}...")
    
    deployed_tools.append({
        'tool_name': tool_name,
        'function_name': function_name,
        'lambda_arn': lambda_arn,
        'tool_definition': tool_definition
    })

print(f"\n✓ Deployed {len(deployed_tools)} tool Lambdas")

# Step 2: Register tools as Gateway targets

created_targets = []

for tool in deployed_tools:
    print(f"\n  Registering {tool['tool_name']}...")
 
    try:
        response = gateway_client.create_gateway_target(
            gatewayIdentifier=GATEWAY_ID,
            name=f"{tool['tool_name'].replace('_', '-')}-target",
            targetConfiguration={
                "mcp": {
                    "lambda": {
                        "lambdaArn": tool["lambda_arn"],
                        "toolSchema": {
                            "inlinePayload": [
                                tool["tool_definition"]
                            ]
                        }
                    }
                }
            },
            credentialProviderConfigurations=[
                {
                    "credentialProviderType": "GATEWAY_IAM_ROLE"
                }
            ]
        )

        status_code = response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        if status_code not in [200, 202]:
            print(f"    ✗ Failed to create target: {response.status_code}")
            print(f"    Response: {response.text}")
            continue
        
        # target_data = response.json()
        target_id = response['targetId']
        print(f"    ✓ Target created: {target_id}")
        
        # Wait for target to be READY using signed requests
        print(f"    Waiting for target to be READY...")
        
        for attempt in range(18):  # 3 minutes max
            try:
                response = gateway_client.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=target_id)
                status_code = response.get("ResponseMetadata", {}).get("HTTPStatusCode")

                
                if status_code == 200:
                    # status_data = status_response.json()
                    status = response.get('status', 'UNKNOWN')
                    
                    if status == 'READY':
                        print(f"    ✓ Target is READY")
                        created_targets.append({
                            'tool_name': tool['tool_name'],
                            'target_id': target_id,
                            'lambda_arn': tool['lambda_arn'],
                            'status': 'READY'
                        })
                        break
                    elif status == 'FAILED':
                        print(f"    ✗ Target FAILED")
                        print(f"    Failure details: {json.dumps(response, indent=6, default=str)}")
                        break
                else:
                    print(f"    Status check error: HTTP {status_code}")
                
            except Exception as e:
                print(f"    Status check error: {e}")
            
            time.sleep(10)
            
    except Exception as e:
        print(f"    ✗ Failed to create target: {e}")

# Summary
print(f"\n{'='*80}")
print(f"Summary:")
print(f"  • Tool Lambdas deployed: {len(deployed_tools)}")
print(f"  • Gateway targets created: {len(created_targets)}")
print(f"{'='*80}\n")

for target in created_targets:
    print(f"  ✓ {target['tool_name']}: {target['target_id']} ({target['status']})")

if len(created_targets) < len(deployed_tools):
    print(f"\n⚠ Warning: Not all targets were created successfully")
    print(f"  Some tools may not be available through the Gateway")
else:
    print(f"\n✅ All tools are registered and ready!")

# Store for cleanup
DEPLOYED_TOOL_FUNCTIONS = [t['function_name'] for t in deployed_tools]
CREATED_TARGET_IDS = [t['target_id'] for t in created_targets]


---

## Part 2: Testing

### Step 2.1: Test with Different Client IDs
Verify each client sees only their permitted tools.

In [ ]:
# Test with different client IDs to verify fine-grained access control
print("="*80)
print("Testing Fine-Grained Access Control with Different Clients")
print("="*80)

# Define test clients with their expected permissions
test_clients = [
    {
        'name': 'full-access',
        'client_id': CLIENT_ID_FULL,
        'expected_tools': ['weather_tool', 'database_query_tool', 'calculation_tool', 'search_tool', 'file_handler_tool']
    },
    {
        'name': 'readonly',
        'client_id': CLIENT_ID_READONLY,
        'expected_tools': ['weather_tool', 'search_tool']
    },
    {
        'name': 'calculator',
        'client_id': CLIENT_ID_CALCULATOR,
        'expected_tools': ['calculation_tool']
    },
    {
        'name': 'data',
        'client_id': CLIENT_ID_DATA,
        'expected_tools': ['database_query_tool', 'file_handler_tool', 'calculation_tool']
    }
]

# Get client secrets for each client
print("\nRetrieving client secrets from Cognito...")
client_secrets = {}
for client_config in test_clients:
    try:
        response = cognito_client.describe_user_pool_client(
            UserPoolId=USER_POOL_ID,
            ClientId=client_config['client_id']
        )
        client_secrets[client_config['client_id']] = response['UserPoolClient']['ClientSecret']
        print(f"  ✓ Retrieved secret for {client_config['name']}")
    except Exception as e:
        print(f"  ✗ Failed to get secret for {client_config['name']}: {e}")

# Test each client
test_results = []

for client_config in test_clients:
    print(f"\n{'='*60}")
    print(f"Testing Client: {client_config['name']}")
    print(f"{'='*60}")
    print(f"  Client ID: {client_config['client_id']}")
    print(f"  Expected tools: {client_config['expected_tools']}")
    
    client_id = client_config['client_id']
    client_secret = client_secrets.get(client_id)
    
    if not client_secret:
        print(f"  ✗ No client secret available, skipping")
        test_results.append({'name': client_config['name'], 'passed': False, 'reason': 'No secret'})
        continue
    
    try:
        # Step 1: Get access token using TOKEN_URL
        print("\n  Step 1: Requesting access token...")
        print(f"  Token URL: {TOKEN_URL}")
        time.sleep(2)  # Brief pause
        
        token_request_response = requests.post(
            TOKEN_URL,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            data={
                "grant_type": "client_credentials",
                "client_id": client_id,
                "client_secret": client_secret,
                "scope": "gateway/tools"
            }
        )
        
        if token_request_response.status_code != 200:
            print(f"    ✗ Token request failed: {token_request_response.status_code}")
            print(f"    Response: {token_request_response.text}")
            test_results.append({'name': client_config['name'], 'passed': False, 'reason': 'Token failed'})
            continue
        
        token_data = token_request_response.json()
        token = token_data['access_token']
        print(f"    ✓ Token obtained (expires in {token_data.get('expires_in')}s)")
        
        # Step 2: Call Gateway to list tools using MCP protocol
        print("\n  Step 2: Calling Gateway to list tools...")
        
        # MCP tools/list request
        mcp_request = {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/list",
            "params": {}
        }
        
        response = requests.post(
            GATEWAY_URL,
            headers={
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json"
            },
            json=mcp_request
        )
        
        if response.status_code != 200:
            print(f"    ✗ Gateway request failed: {response.status_code}")
            print(f"    Response: {response.text}")
            test_results.append({'name': client_config['name'], 'passed': False, 'reason': f'HTTP {response.status_code}'})
            continue
        
        result = response.json()
        
        if 'error' in result:
            print(f"    ✗ MCP error: {result['error']}")
            test_results.append({'name': client_config['name'], 'passed': False, 'reason': 'MCP error'})
            continue
        
        # Extract tool names and parse them
        tools = result.get('result', {}).get('tools', [])
        full_tool_names = [tool['name'] for tool in tools]
        
        # Extract just the tool name part after '___' if present
        # Format: "target-name___tool_name" -> "tool_name"
        actual_tool_names = []
        for full_name in full_tool_names:
            if '___' in full_name:
                # Extract the part after ___
                tool_name = full_name.split('___')[1]
            else:
                tool_name = full_name
            actual_tool_names.append(tool_name)
        
        print(f"    ✓ Received {len(actual_tool_names)} tools")
        print(f"    Full names: {full_tool_names}")
        print(f"    Parsed names: {actual_tool_names}")
        
        # Step 3: Verify permissions
        print("\n  Step 3: Verifying permissions...")
        
        expected_tools = set(client_config['expected_tools'])
        actual_tools = set(actual_tool_names)
        
        print(f"    Expected: {sorted(expected_tools)}")
        print(f"    Actual:   {sorted(actual_tools)}")
        
        if expected_tools == actual_tools:
            print(f"\n  ✅ PASS: Client has correct permissions")
            test_results.append({'name': client_config['name'], 'passed': True})
        else:
            print(f"\n  ❌ FAIL: Permission mismatch")
            
            missing = expected_tools - actual_tools
            if missing:
                print(f"    Missing tools: {sorted(missing)}")
            
            extra = actual_tools - expected_tools
            if extra:
                print(f"    Extra tools: {sorted(extra)}")
            
            test_results.append({'name': client_config['name'], 'passed': False, 'reason': 'Mismatch'})
    
    except Exception as e:
        print(f"\n  ✗ Test failed with exception: {e}")
        import traceback
        traceback.print_exc()
        test_results.append({'name': client_config['name'], 'passed': False, 'reason': str(e)})

# Summary
print(f"\n{'='*80}")
print("Test Summary")
print(f"{'='*80}")

passed_count = sum(1 for r in test_results if r['passed'])
total_count = len(test_results)

for result in test_results:
    status = "✅ PASS" if result['passed'] else "❌ FAIL"
    reason = f" ({result.get('reason', '')})" if not result['passed'] and 'reason' in result else ""
    print(f"  {status}: {result['name']}{reason}")

print(f"\nTotal: {passed_count}/{total_count} passed")

if passed_count == total_count:
    print("\n🎉 All tests passed! Fine-grained access control is working correctly.")
else:
    print("\n⚠️  Some tests failed. Check the logs above for details.")

---

## Part 3: Cleanup

⚠️ **WARNING: This will DELETE all resources created in Part 1!**

Only run this section if you want to clean up everything.

### Step 3.1: Delete Gateway

In [ ]:
# Delete Gateway and Targets using Boto3
print("\nDeleting Gateway and targets...")

try:
    # First delete all targets
    if 'CREATED_TARGET_IDS' in globals() and CREATED_TARGET_IDS:
        print(f"  Deleting {len(CREATED_TARGET_IDS)} targets...")
        
        for target_id in CREATED_TARGET_IDS:
            try:
                gateway_client.delete_gateway_target(
                    gatewayIdentifier=GATEWAY_ID,
                    targetId=target_id
                )
                print(f"    ✓ Target deleted: {target_id}")
            except ClientError as e:
                print(f"    ⚠ Error deleting target {target_id}: {e}")
        
        time.sleep(5)  # Wait for targets to be deleted
    
    # Then delete Gateway
    try:
        gateway_client.delete_gateway(gatewayIdentifier=GATEWAY_ID)
        print(f"  ✓ Gateway deleted: {GATEWAY_ID}")
    except ClientError as e:
        print(f"  ⚠ Error deleting gateway: {e}")
        
except Exception as e:
    print(f"  ⚠ Error during Gateway cleanup: {e}")

### Step 3.2: Delete Lambda Functions

In [ ]:
# Delete Lambda functions (interceptor + tools)
print("\nDeleting Lambda functions...")

# Delete interceptor Lambda
try:
    lambda_client.delete_function(FunctionName=LAMBDA_FUNCTION_NAME)
    print(f"  ✓ Interceptor Lambda deleted: {LAMBDA_FUNCTION_NAME}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print(f"  ⚠ Interceptor Lambda not found: {LAMBDA_FUNCTION_NAME}")
    else:
        print(f"  ⚠ Error: {e}")

# Delete tool Lambdas
if 'DEPLOYED_TOOL_FUNCTIONS' in globals() and DEPLOYED_TOOL_FUNCTIONS:
    print(f"\n  Deleting {len(DEPLOYED_TOOL_FUNCTIONS)} tool Lambdas...")
    for function_name in DEPLOYED_TOOL_FUNCTIONS:
        try:
            lambda_client.delete_function(FunctionName=function_name)
            print(f"    ✓ Deleted: {function_name}")
        except ClientError as e:
            if e.response['Error']['Code'] == 'ResourceNotFoundException':
                print(f"    ⚠ Not found: {function_name}")
            else:
                print(f"    ⚠ Error deleting {function_name}: {e}")

### Step 3.3: Delete IAM Roles

In [ ]:
# Delete IAM roles (Lambda interceptor, tools, Gateway)
print("\nDeleting IAM roles...")

# Delete Lambda interceptor role
try:
    print("  Deleting Lambda interceptor role...")
    
    # Detach policies
    iam_client.detach_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
    )
    
    # Delete inline policies
    iam_client.delete_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyName='DynamoDBAccess'
    )
    
    # Delete role
    iam_client.delete_role(RoleName=LAMBDA_ROLE_NAME)
    print(f"    ✓ Lambda interceptor role deleted: {LAMBDA_ROLE_NAME}")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'NoSuchEntity':
        print(f"    ⚠ Role not found: {LAMBDA_ROLE_NAME}")
    else:
        print(f"    ⚠ Error: {e}")

# Delete tool Lambda role
if 'TOOL_ROLE_NAME' in globals():
    try:
        print("  Deleting tool Lambda role...")
        
        # Detach policies
        iam_client.detach_role_policy(
            RoleName=TOOL_ROLE_NAME,
            PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
        )
        
        # Delete role
        iam_client.delete_role(RoleName=TOOL_ROLE_NAME)
        print(f"    ✓ Tool Lambda role deleted: {TOOL_ROLE_NAME}")
        
    except ClientError as e:
        if e.response['Error']['Code'] == 'NoSuchEntity':
            print(f"    ⚠ Role not found: {TOOL_ROLE_NAME}")
        else:
            print(f"    ⚠ Error: {e}")

# Delete Gateway role
if 'GATEWAY_ROLE_NAME' in globals():
    try:
        print("  Deleting Gateway role...")
        
        # Detach admin policy
        iam_client.detach_role_policy(
            RoleName=GATEWAY_ROLE_NAME,
            PolicyArn='arn:aws:iam::aws:policy/AdministratorAccess'
        )
        
        # Delete role
        iam_client.delete_role(RoleName=GATEWAY_ROLE_NAME)
        print(f"    ✓ Gateway role deleted: {GATEWAY_ROLE_NAME}")
        
    except ClientError as e:
        if e.response['Error']['Code'] == 'NoSuchEntity':
            print(f"    ⚠ Role not found: {GATEWAY_ROLE_NAME}")
        else:
            print(f"    ⚠ Error: {e}")

### Step 3.4: Delete DynamoDB Table

In [ ]:
# Delete DynamoDB table
print("\nDeleting DynamoDB table...")

try:
    dynamodb_client.delete_table(TableName=DYNAMODB_TABLE_NAME)
    print(f"✓ DynamoDB table deleted: {DYNAMODB_TABLE_NAME}")
    
    # Wait for deletion
    print("  Waiting for table deletion...")
    waiter = dynamodb_client.get_waiter('table_not_exists')
    waiter.wait(TableName=DYNAMODB_TABLE_NAME)
    print("  ✓ Table deletion complete")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print(f"⚠ Table not found: {DYNAMODB_TABLE_NAME}")
    else:
        print(f"⚠ Error: {e}")


Deleting DynamoDB table...
✓ DynamoDB table deleted: AgentToolPermissions
  Waiting for table deletion...
  ✓ Table deletion complete


---

# Summary

This notebook completed the full lifecycle:

1. ✅ **Setup** - Created DynamoDB, Lambda, IAM Role, and Gateway
2. ✅ **Test** - Verified tool filtering through real Gateway
3. ✅ **Cleanup** - Deleted all resources

## What We Demonstrated

- **Agent-based tool filtering** using DynamoDB permissions
- **Lambda RESPONSE interceptor** that modifies Gateway responses
- **Custom header propagation** (Agent-ID) through the request chain
- **Complete resource lifecycle** management

## Next Steps

- Run again with different configurations
- Add more custom agents and tools
- Integrate with real AgentCore Runtime agents
- Monitor CloudWatch logs for debugging